[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
import torch
import torch.nn as nn
import math

In [18]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    d_k = Q.shape[-1]
    seq_q = Q.shape[-2]
    seq_k = K.shape[-2]
    mask = torch.tril(torch.ones(seq_q, seq_k)).bool()
    attn_score = torch.einsum('...qd,...kd->...qk',Q,K)/math.sqrt(d_k)
    attn_score = attn_score.masked_fill(~mask,-float('inf'))
    return torch.einsum('...qk,...kd->...qd',torch.softmax(attn_score,dim=-1),V)

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.d_model,self.num_heads,self.num_kv_heads = d_model,num_heads,num_kv_heads
        self.d_k = d_model//num_heads
        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model,num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model,d_model)
        

    def forward(self, x):
        q,k,v=self.W_q(x),self.W_k(x),self.W_v(x)
        seq = x.shape[1]
        q = q.view(-1,seq,self.num_heads,self.d_k).transpose(1,2).contiguous()
        k = k.view(-1,seq,self.num_kv_heads,self.d_k).transpose(1,2).contiguous()
        v = v.view(-1,seq,self.num_kv_heads,self.d_k).transpose(1,2).contiguous()
        # expand k,v heads
        num_repeats = self.num_heads//self.num_kv_heads
        k = torch.repeat_interleave(k,repeats=num_repeats,dim=1)
        v = torch.repeat_interleave(v,repeats=num_repeats,dim=1)
        attn = causal_attention(q,k,v)
        return attn.transpose(1,2).contiguous().view(-1,seq,self.d_model)

In [19]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [20]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (3.9ms)
  ✅ [2/5] nn.Linear with correct shapes (0.4ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (2.6ms)
  ✅ [4/5] KV heads are shared correctly (4.1ms)
  ✅ [5/5] Gradient flow (4.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (15.6ms total)
  Progress saved. Run status() to see your dashboard.

